# E7 Forecasts on a T4

Runs the weather model for every E7 cutoff and saves the daily index
values over California. See [docs/multi-day-risk.md](../docs/multi-day-risk.md)
for the frozen specification.

Run on Kaggle with **Accelerator: GPU T4 x2** and **Internet: on**. The
full run needs about 5 GPU-hours, so use **Save Version → Save & Run All**,
which keeps running after the browser closes. The script skips cutoffs
that are already saved.

The notebook gets the project from GitHub. To use a local copy instead,
upload the repository as a Kaggle dataset named `earth-intelligence`
and add it to the notebook.

SFNO runs out of memory on a T4 when its per-pixel layers process the
whole 721 × 1440 grid at once. The script runs those layers in blocks of
64 latitude rows, which gives the same result with less memory.

Output: `/kaggle/working/e7_forecasts.zip`. Unzip it into
`ml/artifacts/e7_forecasts/`.

In [ ]:
!pip install -q "makani @ git+https://github.com/NVIDIA/makani.git@b38fcb2799d7dbc146fa60459f3f9823394a8bf1" "earth2studio[data,sfno]==0.20.0" python-dotenv 2>&1 | tail -3

In [ ]:
import shutil
import subprocess
from pathlib import Path

REPOSITORY = "https://github.com/khoahonguyen0210/Earth_intelligence.git"
INPUT = Path("/kaggle/input")
PROJECT = Path("/kaggle/working/earth-intelligence")
OUTPUT = Path("/kaggle/working/e7_forecasts")
MODEL = "SFNO"


def uploaded_project():
    for marker in INPUT.rglob("ml/pyproject.toml"):
        return marker.parent.parent
    for archive in INPUT.rglob("earth-intelligence.tar.gz"):
        shutil.unpack_archive(archive, PROJECT)
        return PROJECT
    return None


if not PROJECT.exists():
    source = uploaded_project()
    if source is None:
        subprocess.run(
            ["git", "clone", "--depth", "1", REPOSITORY, str(PROJECT)], check=True
        )
    elif source != PROJECT:
        shutil.copytree(source, PROJECT)

## Short check

One cutoff first, to confirm the model loads and the output looks right.

In [ ]:
!cd {PROJECT} && PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True PYTHONPATH=ml/src python scripts/research/generate_e7_forecasts.py --model {MODEL} --output {OUTPUT} --limit 1
!cat {OUTPUT}/metadata.json

## Full run

If the short check took more than 5 minutes, all cutoffs would not
finish in one 12-hour run, so only every other cutoff is kept. This
rule is part of the frozen specification.

In [ ]:
import json

seconds = json.loads((OUTPUT / "metadata.json").read_text())["seconds"]
first = next(iter(seconds.values()))
EVERY = 1 if first <= 300 else 2
print(f"short check: {first} s, keeping every {EVERY} cutoff(s)")
!cd {PROJECT} && PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True PYTHONPATH=ml/src python scripts/research/generate_e7_forecasts.py --model {MODEL} --output {OUTPUT} --every {EVERY}

In [ ]:
shutil.make_archive("/kaggle/working/e7_forecasts", "zip", OUTPUT)
len(list(OUTPUT.glob("forecast_*.npz")))